In [1]:
import masknmf
import torch
import sys
import numpy as np

from typing import *
import fastplotlib as fpl
import masknmf
import scipy
import h5py
import scipy.sparse
import numpy as np
import torch
import os
import roicat
from masknmf.multisession import RoicatDataAdapter, RoicatTracker, RoicatTrackingResults
from functools import partial


%load_ext autoreload
%matplotlib inline

rendercanvas could not load some backends:
terminal: No module named 'blessed'
Rendercanvas selected anywidget backend because running on Jupyter.
To silence this warning, use a fully namespaced name.


# Get a list of masknmf demixing results .hdf5 files on which you'd like to run multisession tracking. Each hdf5 file corresponds to the demixing results from a single session of imaging

In [ ]:
file_list = ["file_1.hdf5",
             "file_2.hdf5",
             "file_3.hdf5"]

# Build a roicat adapter, this puts the data into a format that ROICaT expects

In [ ]:
roicat_input = RoicatDataAdapter.from_masknmf(file_list) 

# Run tracking, either with default parameters or custom parameters

In [ ]:
tracker = RoicatTracker() # Default params. Pass in params=... as specified by ROICaT's documentation to do fancier stuff here


tracking_results = tracker.run_tracking(roicat_input)

# Write the tracking results out to disk. Put the tracking results into a dedicated folder

In [ ]:
tracking_results.to_roicat_dir("./roiat_results_take1")

## To read the results back, do the following:

In [ ]:
tracking_results = masknmf.multisession.RoicatTrackingResults.from_roicat_dir("./roicat_results_take1")

# Visualize results. This uses fastplotlib NDWidget. It's very easy to add arbitrary panels to this visualization, including stimulus traces, other videos, etc. 

In [ ]:
multisess_vis = masknmf.MultiSessionDemixingVis(tracking_results,
                                      device='cuda') ## device = 'cpu' if you have too many movies to fit on GPU



## The code above gives a visualization system that can automatically synchronize data. Below is an example of how to add some stimulus traces and view them with the above plot

In [ ]:
stim_names = ["Stim Sess 1"]
ndw_stim_traces = fpl.NDWidget(
        ref_ranges=multisess_vis.reference_ranges, ## This does the synchronization
        ref_index=multisess_vis.reference_index, ## So does this
        names=[*stim_names],
        controller_ids=[
            tuple(stim_names),
        ],
        size=(600, 300),
    )

num_frames = multisess_vis.ac_arrays[0].shape[0]

## Some random stim onset data
trace = np.zeros((num_frames,))
trace[300:350] = 1.0
ndw_stim_traces[stim_names[0]].add_nd_timeseries(fpl.utils.heatmap_to_positions(trace[None,:], np.arange(trace.shape[0])),
                                                 ("l", "time", "d"),
                                                 ("l", "time", "d"),
                                                 slider_dim_transforms=None, ## Put the per frame timings relative to start of session here
                                                 max_display_datapoints=5000, ## Make this smaller to zoom
                                                 x_range_mode="auto",
                                                 display_window=None,
                                                 name=stim_names[0],
                                                )

## Add a vertical line to show stimulus onset, can do the same with stimulus conclusion
ndw_stim_traces.figure[0].add_inf_line(
    np.array([250]),
    axis="x",
    colors="gray",
    thickness=2,
    dash_pattern="--",
)



## Finally, you can view these different widgets together. Everything is synchronized in time here

In [ ]:
from ipywidgets import HBox, VBox
VBox([multisess_vis.show(), ndw_stim_traces.show()])